# C3 exp10, the missing run: 2 examples per label, options rotated by 5

exp10 scored two settings under the 7 rotations of the options. One run, `c3_exp10_7b_shots2_r5`, stopped at its prefix-cache check: its fixed allowance (0.01) was too tight when the checked prompts needed no padding. The allowance is now 0.05 (`src/smi/prompting/backend.py`, `CHECK_FLOOR`). This notebook runs that one run again, with exactly the settings of the other six.

The 2-per-label averaging and PriDe need no GPU: they are computed afterwards from the seven runs' scores (`scripts/pride_prompting.py`).

Note: `tasks/C3-prompting-baseline/experiments/exp10_pride-7b/NOTE.md`. Temporary trial (`scripts/trial_prompting.py`); not reportable (PROTOCOL §4.1).

Before running, in the notebook settings: **GPU T4 x2** (one GPU is used), **Internet on**, the `smi-v3-meetings` dataset attached.

Expected time: about 20 minutes (model download and loading, then ~15 min of scoring).

## 1. GPU

In [ ]:
!nvidia-smi

## 2. Code and packages
`bitsandbytes` is needed for the 4-bit 7B model. Do **not** install `requirements.txt` here.

In [ ]:
!rm -rf /kaggle/working/repo
!git clone -b C3/prompting-baseline https://github.com/ahmedwaleedaref/Meeting-Intelligent-System-.git /kaggle/working/repo
%cd /kaggle/working/repo
!git log --oneline -3
!pip install -q transformers==5.20.1 bitsandbytes

## 3. Data folder: the v3 one

In [ ]:
import glob, json, os, subprocess, time

hits = [h for h in glob.glob("/kaggle/input/**/source.json", recursive=True) if json.load(open(h))["data_version"] == "v3"]
assert len(hits) == 1, f"expected exactly one v3 dataset under /kaggle/input, found {hits} : is smi-v3-meetings attached?"
DATA_ROOT = os.path.dirname(hits[0])
print("DATA_ROOT =", DATA_ROOT, "|", json.load(open(hits[0])))
os.makedirs("/kaggle/working/logs", exist_ok=True)

## 4. The run (~20 min)
The log starts with the `prefix cache :` check line; the progress lines follow.

In [ ]:
RUN = "c3_exp10_7b_shots2_r5"
command = ["python", "scripts/trial_prompting.py", "--data-root", DATA_ROOT, "--out-dir", "/kaggle/working/runs",
           "--run-name", RUN, "--model", "Qwen/Qwen2.5-7B-Instruct", "--load-in-4bit", "--prefix-cache", "--batch-size", "32",
           "--shots-per-class", "2", "--shot-seed", "0", "--option-shift", "5"]
with open(f"/kaggle/working/logs/{RUN}.log", "w") as log:
    done = subprocess.run(command, cwd="/kaggle/working/repo", env=dict(os.environ, CUDA_VISIBLE_DEVICES="0", PYTHONPATH="src"),
                          stdout=log, stderr=subprocess.STDOUT)
print("\n".join(l for l in open(f"/kaggle/working/logs/{RUN}.log").read().splitlines() if "Loading weights" not in l)[-1500:])
print("exit code", done.returncode)

## 5. Download the output
Right-hand panel: **Output** → `/kaggle/working` → `c3_exp10_r5_runs.zip` → **⋮** → **Download**.

In [ ]:
%cd /kaggle/working
!zip -qr c3_exp10_r5_runs.zip runs logs
!ls -la c3_exp10_r5_runs.zip